# Unstable gradients

In a deep network the gradient reaching the first layer is a product of many factors, one per layer. If each factor is a little below 1 the product vanishes. If each is a little above 1 it explodes. Either way the early layers stop learning, or the loss turns to `nan`. This notebook measures the gradient per layer, then fixes it with initialisation, activation choice, batch norm and gradient clipping.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import set_seed, setup_keras, weight_gradient_norms

setup_keras("torch")
import keras
import matplotlib.pyplot as plt
import numpy as np
import torch

from common.datasets import load_mnist

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Pass a message down a line of 10 people, each of whom repeats it at 90% volume. The last person hears 35% of it. Make each repeat at 50% and the last person hears 0.1%. Now make it 150% and the last person is deafened by 58 times the original.

Backprop sends the gradient the same way, one layer at a time. The per-layer factor is set by the weights and by the slope of the activation. Get either wrong and depth becomes a liability.

## 2. Maths and cost function

The gradient at layer $l$ in a depth-$L$ network contains the product

$$\prod_{k=l+1}^{L}W_k^\top\,\mathrm{diag}\big(\sigma'(\mathbf z_k)\big)$$

The sigmoid slope $\sigma'(z)=\sigma(1-\sigma)$ is at most 0.25, so ten sigmoid layers shrink the signal by at least $0.25^{10}\approx10^{-6}$ before the weights count. ReLU has slope 1 when active, so it does not shrink the signal by itself.

Initialisation aims to keep the variance of activations constant from layer to layer. For a layer with $n_{in}$ inputs and $n_{out}$ outputs:

$$\text{Glorot: }\mathrm{Var}(W)=\frac{2}{n_{in}+n_{out}}\qquad \text{He: }\mathrm{Var}(W)=\frac{2}{n_{in}}$$

Glorot suits tanh and sigmoid. He adds a factor of 2 because ReLU zeroes half the activations. The cost function is the usual cross-entropy. Nothing about it changes. The problem is how the gradient travels.

## 3. From scratch

Push random data through 30 ReLU layers of 100 units in NumPy, and watch the standard deviation of the activations at the end for four weight scales.

In [ ]:
x = rng.normal(size=(500, 100))
inits = {
    "std 0.01": lambda n_in, n_out: 0.01,
    "Glorot": lambda n_in, n_out: np.sqrt(2 / (n_in + n_out)),
    "He": lambda n_in, n_out: np.sqrt(2 / n_in),
    "std 0.3": lambda n_in, n_out: 0.3,
}
for name, scale in inits.items():
    h = x
    for _ in range(30):
        h = np.maximum(0, h @ rng.normal(0, scale(100, 100), (100, 100)))
    print(f"{name:>9}: activation std after 30 layers = {h.std():.3g}")
print("sigmoid slope bound over 10 layers:", 0.25**10)

Only He keeps the signal near 1 (1.4 after 30 layers). Glorot loses a factor of about $\sqrt2$ per layer with ReLU, which compounds to $10^{-5}$. Std 0.01 vanishes to $10^{-35}$ and std 0.3 explodes to $10^{10}$. The forward activations and the backward gradients follow the same rule, so a signal that dies going forward starves the gradient too.

## 4. Library version

A 10-layer, 50-unit MLP on MNIST, built with different activation and initialiser pairs. `weight_gradient_norms` from `common.dl` runs one batch and returns the gradient norm for each weight matrix. Plot them by layer, input end first.

In [ ]:
x_tr, y_tr, x_te, y_te = load_mnist(3000, 1000, seed=SEED)
x_tr, x_te = x_tr.reshape(len(x_tr), -1), x_te.reshape(len(x_te), -1)
y_tr, y_te = y_tr.astype("int64"), y_te.astype("int64")


def deep_net(activation="relu", init="he_normal", batch_norm=False, optimizer=None) -> keras.Model:
    set_seed(SEED)
    layers = [keras.Input((784,))]
    for _ in range(10):
        layers.append(keras.layers.Dense(50, kernel_initializer=init, use_bias=not batch_norm))
        if batch_norm:
            layers.append(keras.layers.BatchNormalization())
        layers.append(keras.layers.Activation(activation))
    layers.append(keras.layers.Dense(10, kernel_initializer=init))  # logits
    model = keras.Sequential(layers)
    model.compile(
        optimizer=optimizer or keras.optimizers.SGD(0.05),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    return model


def layer_norms(model: keras.Model) -> np.ndarray:
    norms = weight_gradient_norms(model, x_tr[:256], y_tr[:256], torch.nn.functional.cross_entropy)
    return np.array(list(norms.values()))


configs = {
    "sigmoid + Glorot": ("sigmoid", "glorot_uniform"),
    "ReLU + Glorot": ("relu", "glorot_uniform"),
    "ReLU + He": ("relu", "he_normal"),
}
for name, (act, init) in configs.items():
    norms = layer_norms(deep_net(act, init))
    plt.semilogy(range(1, 12), norms, marker="o", label=name)
    print(f"{name:>17}: first layer {norms[0]:.1e}, last layer {norms[-1]:.1e}")
plt.xlabel("layer (1 = nearest the input)")
plt.ylabel("gradient norm")
plt.legend()
plt.show()

With sigmoid, the first layer's gradient is about $10^{-6}$ against 0.2 at the output, a drop of more than five orders of magnitude. ReLU with Glorot is flat across layers but small overall (0.04 at the input). He initialisation keeps it near 1. Now train each for 5 epochs and see whether the measurement predicts the result.

In [ ]:
def fit_report(name: str, model: keras.Model, epochs: int = 5) -> None:
    hist = model.fit(x_tr, y_tr, epochs=epochs, batch_size=128, verbose=0).history
    acc = model.evaluate(x_te, y_te, verbose=0)[1]
    print(f"{name:>24}: train loss {hist['loss'][-1]:.3f}, test acc {acc:.3f}")


for name, (act, init) in configs.items():
    fit_report(name, deep_net(act, init))
fit_report("ReLU + He + batch norm", deep_net("relu", "he_normal", batch_norm=True))

The sigmoid net never leaves the starting loss of 2.30 (chance is $\ln10$) and scores 11.5%. ReLU with Glorot barely moves. He reaches about 68%, and adding **batch norm** lifts the loss from 1.00 to 0.61. Batch norm re-centres and re-scales each layer's inputs during training, which keeps the signal in a healthy range whatever the weights did. (The batch-norm gradients are not plotted here because the helper measures the model in inference mode, where batch norm uses its stored statistics.)

## 5. Tuning

Activation choice, at its matching initialiser, with the same net and 5 epochs.

In [ ]:
for act, init in [
    ("sigmoid", "glorot_uniform"),
    ("tanh", "glorot_uniform"),
    ("relu", "he_normal"),
    ("leaky_relu", "he_normal"),
    ("elu", "he_normal"),
    ("selu", "lecun_normal"),
]:
    fit_report(f"{act} + {init}", deep_net(act, init))

Sigmoid is out. Tanh works with Glorot because its slope near zero is 1. ReLU can leave units stuck at zero for good ("dying ReLU"), and leaky ReLU, ELU and SELU avoid that with a small slope or smooth negative side. Here ELU is best at a train loss of 0.35. Differences between the working activations are small next to the gap between working and not working.

## 6. What breaks

Exploding gradients. Set the initial weights too large (std 0.3) and the first-layer gradient norm is already in the hundreds.

In [ ]:
big = keras.initializers.RandomNormal(0, 0.3)
norms = layer_norms(deep_net("relu", big))
print(f"gradient norms, input to output: {norms[0]:.0f} to {norms[-1]:.0f}")
fit_report("std 0.3, no clipping", deep_net("relu", big))
fit_report(
    "std 0.3, clipnorm=1", deep_net("relu", big, optimizer=keras.optimizers.SGD(0.05, clipnorm=1.0))
)

Unclipped, the loss goes to `nan` in the first epoch and accuracy is at chance. With `clipnorm=1.0` the optimiser rescales any gradient whose norm exceeds 1, so training survives and the loss stays finite and ends at 2.2. It is still a bad run (32% accuracy), because clipping stops the explosion without fixing the cause. Clipping suits recurrent networks, where large gradients come in bursts. For a feed-forward net, fix the initialisation first.